In [4]:
# %pip install xmltodict
# %pip install scipy
%pip install --upgrade numexpr

Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
from Corpus import Corpus
from Document import Document
from SearchEngine import SearchEngine

# 1.1 Chargement du fichier CSV
df_discours = pd.read_csv("discours_US.csv", sep='\t', encoding='utf-8')

# 1.2 Vérification des auteurs
print("Distribution des auteurs :")
print(df_discours['speaker'].value_counts())

Distribution des auteurs :
speaker
CLINTON    93
TRUMP      71
Name: count, dtype: int64


In [3]:
# 1.3 Création du corpus SANS lancer la recherche API
# On passe dataframe=True pour entrer dans la nouvelle condition du __init__
mon_corpus = Corpus("Discours US", dataframe=True)

for index, row in df_discours.iterrows():
    # Extraction des données
    nom_auteur = row['speaker']
    texte_complet = str(row['text'])
    
    # Découpage par phrase
    phrases = texte_complet.split('.')
    
    for i, p in enumerate(phrases):
        phrase_nettoyee = p.strip()
        if len(phrase_nettoyee) > 20:
            doc = Document(
                titre=f"Discours {nom_auteur} - ph {i}", 
                auteur=nom_auteur, 
                date=row['date'], 
                url=row['link'], 
                texte=phrase_nettoyee
            )
            # On utilise la méthode add() qu'on a ajoutée
            mon_corpus.add(doc)

# On met à jour la chaîne globale pour le concordancier
mon_corpus.textOnOneLine = mon_corpus.getOneLineOfText()

print(f"Extraction terminée : {mon_corpus.nb_docs} phrases ajoutées.")

Corpus 'Discours US' initialisé vide pour chargement CSV.
Extraction terminée : 28828 phrases ajoutées.


In [4]:
# 2.1 Initialisation du moteur de recherche
print("Initialisation du moteur (calcul de l'indexation TF-IDF)...")
engine = SearchEngine(mon_corpus)
print("Indexation terminée !")

Initialisation du moteur (calcul de l'indexation TF-IDF)...
Indexation terminée !


In [6]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# 3.1 & 3.4 Création des composants
input_text = widgets.Text(placeholder='Mots-clés...', description='Recherche :')
slider_nb = widgets.IntSlider(value=5, min=1, max=20, description='Top :')
bouton_go = widgets.Button(description="Rechercher", button_style='success', icon='search')
zone_resultat = widgets.Output() # Pour afficher les tableaux de résultats

In [7]:
# 3.5 Fonction appelée lors du clic
def executer_recherche(b):
    with zone_resultat:
        clear_output() # Efface les résultats précédents
        requete = input_text.value
        nb = slider_nb.value

        if not requete.strip():
            print("Veuillez entrer un mot.")
            return

        # Appel de la fonction search de ton SearchEngine
        df_res = engine.search(requete, k=nb)

        print(f"Résultats pour '{requete}' :")
        display(df_res)

# On lie le bouton à la fonction
bouton_go.on_click(executer_recherche)

# 3.2 & 3.3 Assemblage de l'interface
layout = widgets.VBox([
    widgets.HBox([input_text, slider_nb]),
    bouton_go,
    zone_resultat
])

display(layout)